In [1]:
%pip install -U sentence-transformers

/Users/fuyao/Desktop/mentor-mentee-matching/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

try:
    mentees = pd.read_csv("Mentees.csv", encoding='latin1')
    mentors = pd.read_csv("Mentors.csv", encoding='latin1')
    print("Mentees Columns:", mentees.columns.tolist())
    print("Mentors Columns:", mentors.columns.tolist())
    print("\nMentees head:\n", mentees.head(2))
    print("\nMentors head:\n", mentors.head(2))
except Exception as e:
    print(f"Error: {e}")

Mentees Columns: ['Id', 'First_name', 'Last_name', 'Country', 'Language', 'International_vs_Domestic_preference', 'Other_preferences', 'objective']
Mentors Columns: ['Id', 'Mentor_name', 'Mentor_email', 'Mentor_Sisid', 'Country', 'International?', 'Gender', 'objective']

Mentees head:
    Id First_name Last_name Country  Language  \
0   1      Layla    Fowler  Brazil   English   
1   2       Minh   Johnson  Brazil  Mandarin   

  International_vs_Domestic_preference Other_preferences  \
0                   A Domestic student               NaN   
1        Same home country if possible               NaN   

                                     objective  
0   Learning how to network at industry events  
1  Adjusting to life in Canada; making friends  

Mentors head:
    Id       Mentor_name                 Mentor_email  Mentor_Sisid   Country  \
0   1  Danielle Johnson  mentor1@schulich-example.ca       1000001  Colombia   
1   2     Joshua Walker  mentor2@schulich-example.ca       10000

In [3]:
### Sample greedy matching logic

# # Mock similarity matrix (Rows = Mentees, Columns = Mentors)
# mock_scores = pd.DataFrame({
#     'Mentor_A': [0.90, 0.20, 0.50],
#     'Mentor_B': [0.40, 0.85, 0.10],
#     'Mentor_C': [0.30, 0.60, 0.95]
# }, index=['Mentee_1', 'Mentee_2', 'Mentee_3'])

# def greedy_match(similarity_matrix):
#     # Make a copy so we don't destroy the original dataset
#     matrix = similarity_matrix.copy()
#     matches = []

#     # Loop continues as long as there are mentees and mentors left
#     while not matrix.empty and not matrix.columns.empty:

#         # 1. Find the maximum value in the entire grid
#         # .stack() turns the grid into a single list, idxmax() finds the top score's labels
#         best_mentee, best_mentor = matrix.stack().idxmax()
#         best_score = matrix.loc[best_mentee, best_mentor]

#         # 2. Record the pair and their score
#         matches.append((best_mentee, best_mentor, best_score))

#         # 3. Drop the matched row and column from the pool
#         matrix = matrix.drop(index=best_mentee, columns=best_mentor)

#     return pd.DataFrame(matches, columns=['Mentee', 'Mentor', 'Score'])

# # Run the function
# final_pairs = greedy_match(mock_scores)
# print(final_pairs)

In [4]:
### testing for P text embeddings using Sentence Transformer

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load a fast, highly-rated pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Grab a sample from our dataset
mentee_text = "Adjusting to life in Canada; Making friends"
mentor_text = "I want to help international students feel at home and build a social network."

# 3. Convert the text into numeric vectors (embeddings)
mentee_embedding = model.encode([mentee_text])
mentor_embedding = model.encode([mentor_text])

# 4. Calculate the cosine similarity (0.0 to 1.0)
similarity_score = cosine_similarity(mentee_embedding, mentor_embedding)[0][0]

print(f"Similarity Score: {similarity_score:.2f}")

/Users/fuyao/Desktop/mentor-mentee-matching/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/Users/fuyao/Desktop/mentor-mentee-matching/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Similarity Score: 0.36


In [5]:
### Generate objective matrix
from sentence_transformers import SentenceTransformer

# 1. Load the pre-trained semantic model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Extract the objectives as lists (using fillna to prevent errors from empty cells)
mentee_objectives = mentees['objective'].fillna("").tolist()
mentor_objectives = mentors['objective'].fillna("").tolist()

# 3. Convert the lists of text into matrices of numeric vectors
mentee_embeddings = model.encode(mentee_objectives)
mentor_embeddings = model.encode(mentor_objectives)

# Check the dimensions of our new vectors
print("Mentee embeddings shape:", mentee_embeddings.shape)
print("Mentor embeddings shape:", mentor_embeddings.shape)

from sklearn.metrics.pairwise import cosine_similarity

# Generates an 18x28 matrix of scores between 0 and 1
objective_scores = cosine_similarity(mentee_embeddings, mentor_embeddings)

/Users/fuyao/Desktop/mentor-mentee-matching/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Mentee embeddings shape: (18, 384)
Mentor embeddings shape: (28, 384)


In [6]:
### Generate background matrix

#Build the 18x28 Background matrix
background_scores = np.array([
    [1 if mentee_country == mentor_country else 0 for mentor_country in mentors['Country']]
    for mentee_country in mentees['Country']
])

In [7]:
### gender inference for mentees

# install the library: pip install gender-guesser
import gender_guesser.detector as gender

# Initialize the detector
d = gender.Detector()

# Create a function to apply to our DataFrame
def guess_gender(name):
    # The library expects capitalized names
    capitalized_name = str(name).capitalize()
    return d.get_gender(capitalized_name)

# Apply it to the First_name column to create a new Gender column
mentees['Gender'] = mentees['First_name'].apply(guess_gender)

print(mentees[['First_name', 'Gender']].head())

# Create the mapping dictionary to normalize the language of 'Gender' between two dfs
gender_mapping = {
    'female': 'Woman',
    'mostly_female': 'Woman',
    'male': 'Man',
    'mostly_male': 'Man',
    'andy': 'Unknown',
    'unknown': 'Unknown'
}

# Apply the dictionary to update the column
mentees['Gender'] = mentees['Gender'].map(gender_mapping)

print(mentees[mentees['Id']== 17][['First_name', 'Gender']])


  First_name         Gender
0      Layla         female
1       Minh  mostly_female
2     Carlos           male
3      Chidi        unknown
4        Ivy         female
   First_name Gender
16       Minh  Woman


In [8]:
### Generate gender matrix

# Define our neutral scoring rules
def score_gender(mentee_gender, mentor_gender):
    if mentee_gender == 'Unknown':
        return 0.5
    elif mentee_gender == mentor_gender:
        return 1.0
    else:
        return 0.0

# Build the 18x28 Gender matrix
gender_scores = np.array([
    [score_gender(mentee_g, mentor_g) for mentor_g in mentors['Gender']]
    for mentee_g in mentees['Gender']
])

In [9]:
### combine these three matrices into a final_scores matrix using 40/30/30 weights
final_scores = (objective_scores * 0.40) + (background_scores * 0.30) + (gender_scores * 0.30)

# Create readable labels using the 'Id' columns
mentee_labels = ["Mentee_" + str(id) for id in mentees['Id']]
mentor_labels = ["Mentor_" + str(id) for id in mentors['Id']]

# Convert the math grid into a labeled DataFrame
similarity_matrix = pd.DataFrame(
    final_scores,
    index=mentee_labels,
    columns=mentor_labels
)

In [10]:
def greedy_match(similarity_matrix):
    # Make a copy so we don't destroy the original dataset
    matrix = similarity_matrix.copy()
    matches = []

    # Loop continues as long as there are mentees and mentors left
    while not matrix.empty and not matrix.columns.empty:

        # 1. Find the maximum value in the entire grid
        # .stack() turns the grid into a single list, idxmax() finds the top score's labels
        best_mentee, best_mentor = matrix.stack().idxmax()
        best_score = matrix.loc[best_mentee, best_mentor]

        # 2. Record the pair and their score
        matches.append((best_mentee, best_mentor, best_score))

        # 3. Drop the matched row and column from the pool
        matrix = matrix.drop(index=best_mentee, columns=best_mentor)

    return pd.DataFrame(matches, columns=['Mentee', 'Mentor', 'Score'])

# Run the function
final_pairs = greedy_match(similarity_matrix)
print(final_pairs)

       Mentee     Mentor     Score
0   Mentee_18   Mentor_2  0.832721
1    Mentee_4  Mentor_27  0.724738
2    Mentee_9   Mentor_1  0.714474
3    Mentee_2   Mentor_3  0.711038
4   Mentee_11  Mentor_11  0.704436
5    Mentee_8  Mentor_15  0.680786
6   Mentee_17  Mentor_12  0.653927
7   Mentee_10  Mentor_19  0.653781
8   Mentee_14  Mentor_10  0.640121
9    Mentee_3   Mentor_5  0.619428
10   Mentee_7   Mentor_4  0.616400
11  Mentee_16  Mentor_20  0.574738
12  Mentee_12   Mentor_9  0.497507
13  Mentee_13  Mentor_17  0.421740
14   Mentee_1  Mentor_21  0.389366
15   Mentee_6  Mentor_18  0.375717
16   Mentee_5  Mentor_16  0.360557
17  Mentee_15  Mentor_13  0.143140


In [11]:
final_pairs['mentee_id'] = final_pairs['Mentee'].str.replace('Mentee_', '').astype(int)
final_pairs['mentor_id'] = final_pairs['Mentor'].str.replace('Mentor_', '').astype(int)

print(final_pairs.head())

      Mentee     Mentor     Score  mentee_id  mentor_id
0  Mentee_18   Mentor_2  0.832721         18          2
1   Mentee_4  Mentor_27  0.724738          4         27
2   Mentee_9   Mentor_1  0.714474          9          1
3   Mentee_2   Mentor_3  0.711038          2          3
4  Mentee_11  Mentor_11  0.704436         11         11
